# 🎙️ Análise de áudio — relato do paciente

Módulo do sistema de monitoramento multimodal de pacientes.

Pipeline completo:
1. **Azure Speech-to-Text** — transcreve arquivo `.wav` para texto
2. **Azure AI Language** — sentimento, key phrases e entidades
3. **Palavras-chave críticas** — alerta se detectar sintomas graves
4. Salva `data/audio/resultado_audio.json` consumido pelo dashboard

> Texto de exemplo simulado — não é de paciente real.

## 1. Dependências

In [ ]:
!pip install -q azure-ai-textanalytics azure-cognitiveservices-speech python-dotenv

## 2. Carregar credenciais

Lê do `.env` local. No Colab, preencha as variáveis diretamente.

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv("../.env")
except Exception:
    pass

SPEECH_KEY        = os.getenv("AZURE_SPEECH_KEY", "")
SPEECH_REGION     = os.getenv("AZURE_SPEECH_REGION", "eastus2")
LANGUAGE_KEY      = os.getenv("AZURE_LANGUAGE_KEY", "")
LANGUAGE_ENDPOINT = os.getenv("AZURE_LANGUAGE_ENDPOINT", "")

# No Colab: descomente e cole suas chaves
# SPEECH_KEY        = "sua_chave_aqui"
# SPEECH_REGION     = "eastus2"
# LANGUAGE_KEY      = "sua_chave_aqui"
# LANGUAGE_ENDPOINT = "https://language-tech-challenge.cognitiveservices.azure.com/"

assert SPEECH_KEY,        "AZURE_SPEECH_KEY não encontrada"
assert LANGUAGE_KEY,      "AZURE_LANGUAGE_KEY não encontrada"
assert LANGUAGE_ENDPOINT, "AZURE_LANGUAGE_ENDPOINT não encontrada"
print("Credenciais carregadas.")

## 3. Selecionar o arquivo de áudio

No Colab, exibe botão de upload. Localmente, procura arquivos em `data/audio/`.
Se nenhum arquivo for encontrado, usa o texto de exemplo simulado.

In [ ]:
WAV_PATH = None
try:
    from google.colab import files  # type: ignore
    print("Faça upload do áudio (wav)...")
    enviado = files.upload()
    if enviado:
        WAV_PATH = next(iter(enviado))
except Exception:
    # fallback local: procura arquivos .wav em data/audio/
    for nome in ["../data/audio/relato.wav", "../data/audio/exemplo.wav"]:
        if os.path.exists(nome):
            WAV_PATH = nome
            break

if WAV_PATH:
    print(f"Áudio selecionado: {WAV_PATH}")
else:
    print("Nenhum arquivo de áudio encontrado — será usado o texto de exemplo simulado.")

## 4. Transcrição com Azure Speech-to-Text

Se um `.wav` foi selecionado, transcreve via Azure. Caso contrário, usa o texto de exemplo.

In [ ]:
import azure.cognitiveservices.speech as speechsdk

TEXTO_EXEMPLO = (
    "Doutor, desde ontem estou com uma dor no peito que vai e volta, "
    "e hoje de manhã senti falta de ar ao subir as escadas. "
    "Também tive um pouco de tontura, mas já melhorou. "
    "Estou muito preocupado e com medo que seja algo grave."
)

def transcrever_wav(wav_path):
    config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, region=SPEECH_REGION)
    config.speech_recognition_language = "pt-BR"
    audio_cfg = speechsdk.audio.AudioConfig(filename=wav_path)
    recognizer = speechsdk.SpeechRecognizer(speech_config=config, audio_config=audio_cfg)
    result = recognizer.recognize_once_async().get()
    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        return result.text
    print(f"Speech-to-Text falhou: {result.reason}")
    return None

if WAV_PATH:
    TEXTO_RELATO = transcrever_wav(WAV_PATH)
    if not TEXTO_RELATO:
        print("Transcrição falhou — usando texto de exemplo.")
        TEXTO_RELATO = TEXTO_EXEMPLO
    else:
        print("Transcrição concluída via Azure Speech-to-Text.")
else:
    TEXTO_RELATO = TEXTO_EXEMPLO

print("\nTexto:")
print(TEXTO_RELATO)

## 5. Conectar ao Azure AI Language

In [ ]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

client = TextAnalyticsClient(
    endpoint=LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(LANGUAGE_KEY),
)
print("Cliente Azure AI Language conectado.")

## 6. Análise de sentimento

In [ ]:
resp_sentimento = client.analyze_sentiment([TEXTO_RELATO], language="pt")
doc = resp_sentimento[0]

sentimento = doc.sentiment
scores = doc.confidence_scores

print(f"Sentimento geral : {sentimento}")
print(f"Confiança        : positivo={scores.positive:.2f} | neutro={scores.neutral:.2f} | negativo={scores.negative:.2f}")

## 7. Extração de key phrases

In [ ]:
resp_kp = client.extract_key_phrases([TEXTO_RELATO], language="pt")
key_phrases = resp_kp[0].key_phrases

print("Key phrases detectadas:")
for kp in key_phrases:
    print(" -", kp)

## 8. Reconhecimento de entidades

In [ ]:
resp_ner = client.recognize_entities([TEXTO_RELATO], language="pt")
entidades = [
    {"texto": e.text, "categoria": e.category, "confianca": round(e.confidence_score, 2)}
    for e in resp_ner[0].entities
]

print("Entidades reconhecidas:")
for e in entidades:
    print(f"  [{e['categoria']}] {e['texto']} (confiança: {e['confianca']})")

## 9. Detecção de palavras-chave críticas

In [ ]:
PALAVRAS_CRITICAS = [
    "dor no peito", "falta de ar", "tontura", "desmaio",
    "dormência", "visão turva", "dor de cabeça forte", "formigamento",
]

texto_lower = TEXTO_RELATO.lower()
alertas_criticos = [p for p in PALAVRAS_CRITICAS if p in texto_lower]

if alertas_criticos:
    print("⚠️  ALERTAS CRÍTICOS detectados:")
    for a in alertas_criticos:
        print(" -", a)
else:
    print("✅ Nenhuma palavra-chave crítica encontrada.")

## 10. Saída padronizada (para a integração)

Salva `data/audio/resultado_audio.json` — consumido automaticamente pelo dashboard Streamlit.

In [ ]:
import json

resultado = {
    "transcricao": TEXTO_RELATO,
    "sentimento": sentimento,
    "confianca_sentimento": {
        "positivo": round(scores.positive, 3),
        "neutro": round(scores.neutral, 3),
        "negativo": round(scores.negative, 3),
    },
    "key_phrases": list(key_phrases),
    "entidades": entidades,
    "alertas_criticos": alertas_criticos,
}

OUT = "../data/audio/resultado_audio.json"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

print(f"Resultado salvo em {OUT}")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

## 11. Conclusão

- **Azure Speech-to-Text** transcreve o relato em `.wav` para texto em português (pt-BR).
- **Azure AI Language** extrai sentimento, entidades e key phrases.
- A detecção de **palavras-chave críticas** complementa a análise semântica.
- O `resultado_audio.json` é consumido pelo dashboard Streamlit para exibir o sentimento e disparar o alerta consolidado.